In [ ]:
import os, random, torch, sys
import matplotlib.pyplot as plt
 
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.optim import lr_scheduler

from torchvision import datasets, models, transforms

cudnn.benchmark = True
plt.ion()   # interactive mode

sys.path.append('../')
from project.resnet_training_utils import train_resnet_architecture, process_and_dump_training_artifacts, cross_val_model_training
random.seed(1234)

## Notebook Purpose
<b> This is notebook number 3a </b>

To give code guidance on training ResNet models used in MixtureModel

### Notebook Order
1. getData
2. downloadData
3. trainResNetModel | trainPromptTransformerClassifier | trainViTClassifier
4. localMixtureEval

In [ ]:
# Get the number of CUDA devices available
num_devices = torch.cuda.device_count()

# Print information about each CUDA device
for i in range(num_devices):
    device_name = torch.cuda.get_device_name(i)
    print(f"Device {i}: {device_name}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataDir = "../data/may_eval_dataset"##where image data is saved. Note -> should be in their class folders

In [ ]:
# Define the transformations to apply to the images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
# Load the images from the directories
image_dataset = datasets.ImageFolder(root=dataDir, transform=transform)
class_names = image_dataset.classes

train_ratio = 0.85

### Set the model architecture 
below, set for 18 or 50

In [ ]:
##Set model architecture
model_ft = models.resnet18(weights='IMAGENET1K_V1')
# model_ft = models.resnet50(weights='IMAGENET1K_V1')

## make sure savepath is related to architecture
save_path = './models/' ##may use something like './models/resnet50/'

In [ ]:
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(device)

criterion = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9, weight_decay = 0.001)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

## Below we want to determine WHAT kind of training we'll do. 

We may train models in various ways. 
1. We may train models vanilla, in which they just train with no additional augmentations of data slices.
2. We may train models with augmentation 
3. We may train models using cross-validation, where we feed the whole image_dataset to the model and it creates n splits of training/validation data allowing the model to be trained on all data.
4. We may also train cross validation WITH augementations, which can prevent overfitting on data. 

In [ ]:
##Vanilla training
model_ft, loss_acc_dict =  train_resnet_architecture(dataset = image_dataset,
    train_ratio= train_ratio, batch_size = 32, model = model_ft,
    criterion = criterion, optimizer_ft = optimizer_ft,
    exp_lr_scheduler = exp_lr_scheduler, patience = 5, num_epochs = 2, 
    augment_transforms = False, base_transforms = transform)      

## Training with Augmentation
# model_ft, loss_acc_dict =  train_resnet_architecture(dataset = image_dataset,
#     train_ratio= train_ratio, batch_size = 32, model = model_ft,
#     criterion = criterion, optimizer_ft = optimizer_ft,
#     exp_lr_scheduler = exp_lr_scheduler, patience = 5, num_epochs = 2, 
#     augment_transforms = True, base_transforms = transform)      

## Cross validation, no augementation
# model_ft, loss_acc_dict = cross_val_model_training(dataset = image_dataset, kfolds = 5, train_ratio = 0.8,
#                               model= model_ft, criterion = criterion, optimizer_ft = optimizer_ft,
#                               exp_lr_scheduler = exp_lr_scheduler, patience = 5, num_epochs = 2)

## Cross validation, with augementation
# model_ft, loss_acc_dict = cross_val_model_training(dataset = image_dataset, kfolds = 5, train_ratio = 0.8,
#                               model = model_ft, criterion = criterion, optimizer_ft = optimizer_ft,
#                               exp_lr_scheduler = exp_lr_scheduler, patience = 5, num_epochs = 2,
#                               augment_transforms = True)

## Save Model

In [ ]:
os.makedirs(save_path, exist_ok = True)
torch.save(model_ft.state_dict(), os.path.join(save_path,'best_model_params.pt'))
process_and_dump_training_artifacts(save_path, loss_acc_dict)